Cell 1 — Config

In [4]:
%run "./00_config.ipynb"

JAVA_HOME: C:\Java\jdk-17
java.exe found at: C:\Java\jdk-17\bin\java.EXE
PySpark home: c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark
bin dir exists: True
['beeline', 'beeline.cmd', 'docker-image-tool.sh', 'find-spark-home', 'find-spark-home.cmd', 'load-spark-env.cmd', 'load-spark-env.sh', 'pyspark', 'pyspark.cmd', 'pyspark2.cmd', 'run-example', 'run-example.cmd', 'spark-class', 'spark-class.cmd', 'spark-class2.cmd', 'spark-connect-shell', 'spark-shell', 'spark-shell.cmd', 'spark-shell2.cmd', 'spark-sql', 'spark-sql.cmd', 'spark-sql2.cmd', 'spark-submit', 'spark-submit.cmd', 'spark-submit2.cmd', 'sparkR', 'sparkR.cmd', 'sparkR2.cmd']
SPARK_HOME env: None
SPARK_HOME: None
JAVA_HOME  : C:\Java\jdk-17
HADOOP_HOME: C:\hadoop
winutils found at: C:\hadoop\bin\winutils.exe
Ready: C:\covid_pipeline\bronze
Ready: C:\covid_pipeline\silver
Ready: C:\covid_pipeline\gold
Spark version: 3.5.3
Spark master : local[*]
Parquet write test succeeded at: C:\covid_pipeline\

In [5]:
# Run 00_config first (or copy its Spark session / path cells above this one)
from pyspark.sql import functions as F
from pyspark.sql.types import *

vacc_schema = StructType([
    StructField("country", StringType(), True),
    StructField("iso_code", StringType(), True),
    StructField("date", StringType(), True),
    StructField("total_vaccinations", DoubleType(), True),
    StructField("people_vaccinated", DoubleType(), True),
    StructField("people_fully_vaccinated", DoubleType(), True),
    StructField("daily_vaccinations_raw", DoubleType(), True),
    StructField("daily_vaccinations", DoubleType(), True),
    StructField("total_vaccinations_per_hundred", DoubleType(), True),
    StructField("people_vaccinated_per_hundred", DoubleType(), True),
    StructField("people_fully_vaccinated_per_hundred", DoubleType(), True),
    StructField("daily_vaccinations_per_million", DoubleType(), True),
    StructField("vaccines", StringType(), True),
    StructField("source_name", StringType(), True),
    StructField("source_website", StringType(), True),
])

df_raw = (
    spark.read
    .option("header", True)
    .schema(vacc_schema)
    .csv(RAW_VACCINATION)
)

print("Raw row count:", df_raw.count())
df_raw.printSchema()

Raw row count: 86512
root
 |-- country: string (nullable = true)
 |-- iso_code: string (nullable = true)
 |-- date: string (nullable = true)
 |-- total_vaccinations: double (nullable = true)
 |-- people_vaccinated: double (nullable = true)
 |-- people_fully_vaccinated: double (nullable = true)
 |-- daily_vaccinations_raw: double (nullable = true)
 |-- daily_vaccinations: double (nullable = true)
 |-- total_vaccinations_per_hundred: double (nullable = true)
 |-- people_vaccinated_per_hundred: double (nullable = true)
 |-- people_fully_vaccinated_per_hundred: double (nullable = true)
 |-- daily_vaccinations_per_million: double (nullable = true)
 |-- vaccines: string (nullable = true)
 |-- source_name: string (nullable = true)
 |-- source_website: string (nullable = true)



In [6]:
# Filter non-country aggregates (OWID sub-national/regional codes: OWID_ENG, OWID_SCT, OWID_WLS, OWID_NIR, OWID_KOS, OWID_CYN, etc.)
df_clean = (
    df_raw
    .filter(~F.col("iso_code").startswith("OWID"))
    .withColumn("date", F.to_date("date", "yyyy-MM-dd"))
    .select(
        "iso_code", "country", "date",
        "total_vaccinations", "people_vaccinated", "people_fully_vaccinated",
        "daily_vaccinations",
        "total_vaccinations_per_hundred",
        F.col("people_vaccinated_per_hundred").alias("vacc_per_100_people"),
        "people_fully_vaccinated_per_hundred",
    )
)

removed = df_raw.count() - df_clean.count()
print(f"Rows removed as non-country aggregates: {removed}")
print("Cleaned row count:", df_clean.count())
df_clean.show(5)

Rows removed as non-country aggregates: 2456
Cleaned row count: 84056
+--------+-----------+----------+------------------+-----------------+-----------------------+------------------+------------------------------+-------------------+-----------------------------------+
|iso_code|    country|      date|total_vaccinations|people_vaccinated|people_fully_vaccinated|daily_vaccinations|total_vaccinations_per_hundred|vacc_per_100_people|people_fully_vaccinated_per_hundred|
+--------+-----------+----------+------------------+-----------------+-----------------------+------------------+------------------------------+-------------------+-----------------------------------+
|     AFG|Afghanistan|2021-02-22|               0.0|              0.0|                   NULL|              NULL|                           0.0|                0.0|                               NULL|
|     AFG|Afghanistan|2021-02-23|              NULL|             NULL|                   NULL|            1367.0|             

In [7]:
df_clean.write.mode("overwrite").parquet(SILVER_VACCINATION)
print("Written to:", SILVER_VACCINATION)
spark.read.parquet(SILVER_VACCINATION).printSchema()

Written to: C:\covid_pipeline\silver\vaccination
root
 |-- iso_code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date: date (nullable = true)
 |-- total_vaccinations: double (nullable = true)
 |-- people_vaccinated: double (nullable = true)
 |-- people_fully_vaccinated: double (nullable = true)
 |-- daily_vaccinations: double (nullable = true)
 |-- total_vaccinations_per_hundred: double (nullable = true)
 |-- vacc_per_100_people: double (nullable = true)
 |-- people_fully_vaccinated_per_hundred: double (nullable = true)

